# Atelier Scikit-learn — Prédiction de l'état des capteurs IoT

Objectif : construire un modèle de Machine Learning capable de prédire automatiquement l'**état** d'un capteur (OK, ALERTE, ERREUR) à partir de ses mesures (température, humidité, pression, consommation).

Workflow suivi : Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement → Modèle → fit() → predict() → Évaluation → Sauvegarde → Chargement → Réutilisation.

## Partie 0 — Mise en place de l'environnement

### 4) Import des librairies

In [4]:
import matplotlib
print(matplotlib.__version__)
import matplotlib.pyplot as plt

3.11.1


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 5)


### 5) Import du dataset dans `df`

In [6]:
df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


### 6) Exploration du dataframe

In [7]:
print("Dimensions :", df.shape)
df.info()

Dimensions : (605, 9)
<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


In [8]:
df.describe(include="all")

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
count,605,605,605,605,599.000000,600.00000,600.000000,600.000000,601
unique,600,600,12,4,NaN,NaN,NaN,NaN,3
top,M0026,2026-01-06 01:00:00,C002,B001,NaN,NaN,NaN,NaN,OK
freq,2,2,52,153,NaN,NaN,NaN,NaN,567
mean,NaN,NaN,NaN,NaN,24.878314,64.92620,1012.221900,208.675417,NaN
std,NaN,NaN,NaN,NaN,4.059576,10.76905,10.599042,72.243567,NaN
min,NaN,NaN,NaN,NaN,-18.500000,28.52000,850.000000,18.120000,NaN
25%,NaN,NaN,NaN,NaN,22.570000,58.17250,1006.790000,160.177500,NaN
50%,NaN,NaN,NaN,NaN,24.860000,65.37500,1012.855000,206.150000,NaN
75%,NaN,NaN,NaN,NaN,27.275000,71.61500,1017.827500,254.127500,NaN


In [9]:
df.isna().sum()

id_mesure       0
date_heure      0
id_capteur      0
batiment        0
temperature     6
humidite        5
pression        5
consommation    5
etat            4
dtype: int64

In [10]:
df["etat"].value_counts()

etat
OK        567
ALERTE     29
ERREUR      5
Name: count, dtype: int64

**Constat :** le dataset contient 605 lignes. La cible `etat` est fortement **déséquilibrée** : très majoritairement `OK`, avec beaucoup moins de cas `ALERTE` et très peu de cas `ERREUR`. On note aussi des valeurs manquantes sur plusieurs colonnes, dont la cible elle-même — à traiter avant la modélisation.

## Partie 1 — Gestion des doublons

### 1) Vérifier l'existence de doublons

In [11]:
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons : {nb_doublons}")
df[df.duplicated()]

Nombre de doublons : 5


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
183,M0599,2026-01-29 22:00:00,C011,B004,22.02,68.09,1005.90,227.81,OK
231,M0026,2026-01-06 01:00:00,C002,B001,22.87,77.99,1010.15,213.19,OK
355,M0147,2026-01-11 02:00:00,C003,B001,26.14,84.97,1003.59,142.31,OK
538,M0456,2026-01-23 23:00:00,C012,B004,20.68,72.69,1022.77,309.01,OK
539,M0302,2026-01-17 13:00:00,C002,B001,22.05,58.26,1007.30,140.42,OK


### 2) Supprimer les doublons et vérifier

In [12]:
df = df.drop_duplicates().reset_index(drop=True)
print("Nombre de doublons après suppression :", df.duplicated().sum())
print("Nouvelles dimensions :", df.shape)

Nombre de doublons après suppression : 0
Nouvelles dimensions : (600, 9)


## Partie 2 — Sélection de y (cible) et X (caractéristiques)

On retire d'abord les lignes où la cible `etat` est manquante : on ne peut pas entraîner ni évaluer un modèle supervisé sans étiquette connue.

In [13]:
df = df.dropna(subset=["etat"]).reset_index(drop=True)
print("Dimensions après suppression des etats manquants :", df.shape)

Dimensions après suppression des etats manquants : (596, 9)


### 1) Définition de X et y

In [15]:
features = ["temperature", "humidite", "pression", "consommation"]
X = df[features]
y = df["etat"]

### 2) Cinq premières lignes de X et de y

In [16]:
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [17]:
y.head()

0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str

### 3) Type du problème de Machine Learning

C'est un problème d'**apprentissage supervisé de classification multi-classes** (3 classes : `OK`, `ALERTE`, `ERREUR`), puisque la cible `etat` est une variable catégorielle discrète et non une valeur numérique continue.